In [1]:
from kaggle.api.kaggle_api_extended import KaggleApi
import pandas as pd

In [2]:
api = KaggleApi()
api.authenticate()  # Requires ~/.kaggle/kaggle.json

We'll use COVID-19 Worldwide Dataset (CSV format):


In [4]:
# Download dataset
api.dataset_download_files('imdevskp/corona-virus-report', path='./data', unzip=True)

Dataset URL: https://www.kaggle.com/datasets/imdevskp/corona-virus-report


In [3]:
# Load into Pandas
df = pd.read_csv('./data/covid_19_clean_complete.csv')
print(df.head())

  Province/State Country/Region       Lat       Long        Date  Confirmed  \
0            NaN    Afghanistan  33.93911  67.709953  2020-01-22          0   
1            NaN        Albania  41.15330  20.168300  2020-01-22          0   
2            NaN        Algeria  28.03390   1.659600  2020-01-22          0   
3            NaN        Andorra  42.50630   1.521800  2020-01-22          0   
4            NaN         Angola -11.20270  17.873900  2020-01-22          0   

   Deaths  Recovered  Active             WHO Region  
0       0          0       0  Eastern Mediterranean  
1       0          0       0                 Europe  
2       0          0       0                 Africa  
3       0          0       0                 Europe  
4       0          0       0                 Africa  


Snowflake Basics (Staging & Loading) 

#Create Snowflake Connection


In [1]:
from snowflake.snowpark import Session

connection_params = {
    "account": "KFNSUIU-DV89729",
    "user": "ARMGHAN",
    "password": "L1f20bsse0414@",
    "role": "ACCOUNTADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "COVID_DB",
    "schema": "PUBLIC"
}

session = Session.builder.configs(connection_params).create()

 Create Database

In [3]:
# Create database
session.sql("CREATE DATABASE IF NOT EXISTS COVID_DB").collect()

[Row(status='COVID_DB already exists, statement succeeded.')]

 Create Internal Stage

In [4]:
# Create internal stage for file uploads
session.sql("CREATE OR REPLACE STAGE COVID_STAGE").collect()

[Row(status='Stage area COVID_STAGE successfully created.')]

Upload to Stage

In [5]:
# Upload CSV to stage
session.file.put(
    './data/covid_19_clean_complete.csv',
    '@COVID_STAGE',
    auto_compress=False
)

[PutResult(source='covid_19_clean_complete.csv', target='covid_19_clean_complete.csv', source_size=3305202, target_size=3305216, source_compression='NONE', target_compression='NONE', status='UPLOADED', message='')]

Create Table 

In [6]:
# Define schema
session.sql("""
CREATE OR REPLACE TABLE COVID_RAW (
    Province_State STRING,
    Country_Region STRING,
    Lat FLOAT,
    Long FLOAT,
    Date TIMESTAMP,
    Confirmed INTEGER,
    Deaths INTEGER,
    Recovered INTEGER,
    Active INTEGER,
    WHO_Region STRING
)
""").collect()


[Row(status='Table COVID_RAW successfully created.')]

Load Data

In [7]:
# Load data from stage
session.sql("""
COPY INTO COVID_RAW
FROM @COVID_STAGE/covid_19_clean_complete.csv
FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1)
""").collect()

[Row(file='covid_stage/covid_19_clean_complete.csv', status='LOADED', rows_parsed=49068, rows_loaded=49068, error_limit=1, errors_seen=0, first_error=None, first_error_line=None, first_error_character=None, first_error_column_name=None)]

*Snowpark* Transformations (ETL)

In [23]:
from snowflake.snowpark.functions import col, to_date

# Convert to Snowpark DataFrame
covid_df = session.table("COVID_RAW")

# 1. Fix date filter (use dataset's actual date range: 2020)
cleaned_df = session.table("COVID_RAW") \
                   .with_column("Date", to_date(col("Date")))\
                   .filter(col("Date") > "2020-01-01")\
                   .dropna()

In [24]:
# 2. Save to table (force overwrite)
cleaned_df.write.mode("overwrite").save_as_table("COVID_CLEANED")

Aggregations (SQL + Python UDF)

In [10]:
from snowflake.snowpark.types import FloatType

# Register UDF for death rate calculation
session.udf.register(
    lambda confirmed, deaths: (deaths / confirmed) * 100 if confirmed > 0 else 0,
    name="CALC_DEATH_RATE",
    return_type=FloatType(),
    input_types=[FloatType(), FloatType()]
)


In [11]:
result = session.sql("""
SELECT 
    Country_Region,
    SUM(Confirmed) AS Total_Confirmed,
    SUM(Deaths) AS Total_Deaths,
    CALC_DEATH_RATE(SUM(Confirmed), SUM(Deaths)) AS Death_Rate
FROM COVID_CLEANED
GROUP BY Country_Region
ORDER BY Total_Confirmed DESC
""").to_pandas()


Automation (Tasks & Streams)

Scheduled Task for Daily Updates

In [12]:
session.sql("""
CREATE OR REPLACE TASK REFRESH_COVID_DATA
WAREHOUSE = COMPUTE_WH
SCHEDULE = 'USING CRON 0 8 * * * UTC'
AS
CALL REFRESH_COVID_PROCEDURE()
""").collect()

[Row(status='Task REFRESH_COVID_DATA successfully created.')]

CDC with Streams (Track Changes)


In [13]:
session.sql("""
CREATE OR REPLACE STREAM COVID_STREAM
ON TABLE COVID_CLEANED
APPEND_ONLY = TRUE
""").collect()

[Row(status='Stream COVID_STREAM successfully created.')]

Advanced Features (Dynamic Tables)

 Dynamic Tables (Auto-Refreshing Aggregations)

In [25]:
session.sql("""
CREATE OR REPLACE DYNAMIC TABLE COVID_AGGREGATED
TARGET_LAG = '1 HOUR'
WAREHOUSE = COMPUTE_WH
AS
SELECT
    Country_Region,
    SUM(Confirmed) AS Total_Confirmed,
    SUM(Deaths) AS Total_Deaths
FROM COVID_CLEANED
GROUP BY Country_Region
""").collect()

[Row(status='Dynamic table COVID_AGGREGATED successfully created.')]

In [26]:
session.table("COVID_CLEANED").count()

14664